# MC MOD Translator

Minecraft の Mod を日本語化するノートブックです。ランタイムのタイプを T4 GPU にしてから、セル 1 から 4 を順に実行してください。

- Mod、翻訳結果、翻訳の途中経過は Google ドライブの `DRIVE_DIR` に保存します。
- ngrok を使う場合は、シークレットに `NGROK_AUTHTOKEN` を登録してください。

## 中断した翻訳を再開する

1. ランタイムに再接続し、セル 1 から 4 をもう一度実行します。
2. `AUTO_RESUME` がオンの場合は自動で再開します。オフの場合は、画面上部の「再開」を押してください。


In [ ]:
#@title 1. 設定
REPO_URL = "https://github.com/wkwk-3/MinecraftMod-AITranslateTool"  #@param {type:"string"}
TUNNEL = "ngrok"  #@param ["ngrok", "cloudflared", "colab"]
#@markdown - ngrok: シークレット `NGROK_AUTHTOKEN` が必要
#@markdown - cloudflared: アカウント登録が不要
#@markdown - colab: ノートブックを開いている本人だけが接続できる
DRIVE_DIR = "MyDrive/mc-mod-translator"  #@param {type:"string"}
#@markdown Google ドライブ上の保存先
AUTO_RESUME = True  #@param {type:"boolean"}
#@markdown 中断した翻訳を起動時に再開する
SAVE_LLAMA_TO_DRIVE = False  #@param {type:"boolean"}
#@markdown オンにすると、llama.cpp (llama-server) を下のドライブのフォルダに保存し、次回からそこから読み込みます (セットアップが短くなります)
LLAMA_DRIVE_DIR = "MyDrive/llama"  #@param {type:"string"}
#@markdown llama-server の保存先・読み込み元 (SAVE_LLAMA_TO_DRIVE がオンのときだけ使います)


In [ ]:
#@title 2. セットアップ (初回は 10〜15 分)
import os, subprocess, shutil
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "GPU が見つかりません。ランタイムのタイプを T4 GPU に変更してください。"
%cd /content
if not os.path.exists("/content/app/run.py"):
    if os.path.exists("/content/mc-mod-translator.zip"):           # ZIP を使う場合は /content/ にアップロードしておく
        !unzip -q -o /content/mc-mod-translator.zip -d /content/app
    else:
        !git clone --depth 1 {REPO_URL} /content/app
%cd /content/app
!pip -q install -r requirements.txt

# 保存先を Google ドライブに置く
from google.colab import drive
drive.mount("/content/drive")
WORKDIR = "/content/drive/" + DRIVE_DIR.strip("/")
os.makedirs(WORKDIR, exist_ok=True)
os.environ["MCT_WORKDIR"] = WORKDIR
os.environ["MCT_CACHEDIR"] = "/content/hf_cache"   # モデルはドライブではなく Colab のディスクに置く
print("保存先:", WORKDIR)

LLAMA_DIR = "/content/llama.cpp"
LLAMA_BIN = f"{LLAMA_DIR}/build/bin/llama-server"
# SAVE_LLAMA_TO_DRIVE がオンのときだけ、ドライブを llama-server の保存先・読み込み元にする。
# ドライブ上では実行権限が保てないため、読み込み時は Colab のディスクにコピー・展開して使う。
#   1. 公式の Linux CUDA 版 (llama-*-bin-ubuntu-cuda-*-x64.tar.gz) があれば展開して試す
#   2. 自分でビルドして保存したもの (llama-server-static) があれば使う
#   3. どちらもない・動かない場合はビルドする (オンなら結果をドライブに保存)
import glob, tarfile
LLAMA_DRIVE = "/content/drive/" + LLAMA_DRIVE_DIR.strip("/")
LLAMA_ON_DRIVE = LLAMA_DRIVE + "/llama-server-static"
USING_PREBUILT = False


def llama_works(binary, env=None):
    r = subprocess.run([binary, "--version"], capture_output=True, text=True, env=env)
    return r.returncode == 0, (r.stderr or r.stdout).strip()[-500:]


if SAVE_LLAMA_TO_DRIVE:
    print("llama-server の保存先・読み込み元:", LLAMA_DRIVE)
    packs = sorted(glob.glob(LLAMA_DRIVE + "/llama-*-bin-ubuntu-cuda-*-x64.tar.gz"))
    if packs:
        PREBUILT_DIR = "/content/llama-prebuilt"
        shutil.rmtree(PREBUILT_DIR, ignore_errors=True)
        os.makedirs(PREBUILT_DIR)
        for pack in [packs[-1]] + sorted(glob.glob(LLAMA_DRIVE + "/cudart-llama-*-bin-ubuntu-cuda-*-x64.tar.gz"))[-1:]:
            with tarfile.open(pack) as t:
                try:
                    t.extractall(PREBUILT_DIR, filter="tar")   # 展開先の外に書き出させない
                except TypeError:
                    t.extractall(PREBUILT_DIR)
        bins = glob.glob(PREBUILT_DIR + "/*/llama-server")
        if bins:
            libdirs = sorted({os.path.dirname(f) for f in glob.glob(PREBUILT_DIR + "/*/*.so*")})
            env = dict(os.environ, LD_LIBRARY_PATH=":".join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]))
            os.chmod(bins[0], 0o755)
            works, msg = llama_works(bins[0], env)
            if works:
                LLAMA_BIN, USING_PREBUILT = bins[0], True
                os.environ["LD_LIBRARY_PATH"] = env["LD_LIBRARY_PATH"]
                print("ドライブの公式 Linux CUDA 版を使います:", os.path.basename(packs[-1]))
            else:
                print("ドライブの公式 Linux CUDA 版は、この Colab では動きません。ビルドします。\n理由:", msg)
    if not USING_PREBUILT and os.path.exists(LLAMA_ON_DRIVE) and not os.path.exists(LLAMA_BIN):
        os.makedirs(os.path.dirname(LLAMA_BIN), exist_ok=True)
        shutil.copy(LLAMA_ON_DRIVE, LLAMA_BIN); os.chmod(LLAMA_BIN, 0o755)
        print("ドライブから読み込みました")
if not USING_PREBUILT:
    !bash setup_llama.sh {LLAMA_DIR}
ok = subprocess.run([LLAMA_BIN, "--version"], capture_output=True, text=True)
if ok.returncode != 0:
    raise SystemExit("llama-server を起動できません:\n" + (ok.stderr or ok.stdout)[-1000:])
# llama-server から GPU が見えているか確認する (見えていないと翻訳が非常に遅くなる)
dev = subprocess.run([LLAMA_BIN, "--list-devices"], capture_output=True, text=True)
dev_out = (dev.stdout + dev.stderr).strip()
print("llama-server が使えるデバイス:\n" + dev_out[-800:])
if "CUDA" not in dev_out:
    print("警告: llama-server から GPU が見えません。ランタイムのタイプが T4 GPU になっているか確認してください。")
if SAVE_LLAMA_TO_DRIVE and not USING_PREBUILT:
    os.makedirs(LLAMA_DRIVE, exist_ok=True)
    if not os.path.exists(LLAMA_ON_DRIVE) or os.path.getsize(LLAMA_ON_DRIVE) != os.path.getsize(LLAMA_BIN):
        shutil.copy(LLAMA_BIN, LLAMA_ON_DRIVE)
        print("ドライブに保存しました")
os.environ["LLAMA_SERVER"] = LLAMA_BIN
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
try:
    from google.colab import userdata
    for k in ("NGROK_AUTHTOKEN", "HF_TOKEN"):
        try: os.environ[k] = userdata.get(k)
        except Exception: pass
except ImportError: pass
print("セットアップが完了しました")

import json, glob
pending = [j for j in (json.load(open(p)) for p in glob.glob(WORKDIR + "/jobs/*/job.json"))
           if j["status"] in ("running", "interrupted", "failed", "cancelled")]
for j in pending:
    u = j["units"].values()
    print(f"中断した翻訳 {j['id']} ({j['status']}): {sum(x['done'] for x in u)}/{sum(x['total'] for x in u)} 件")


In [ ]:
#@title 3. 接続の維持
#@markdown 操作がない状態が続いたときの切断を防ぎます。ブラウザのタブは開いたままにしてください。
#@markdown 無料版の利用時間 (最大約 12 時間) と GPU の利用枠の上限は変わりません。
from IPython.display import display, Javascript
display(Javascript('''
if (!window.__mctKeepAlive) {
  window.__mctKeepAlive = setInterval(() => {
    const b = document.querySelector("colab-connect-button");
    const inner = b && b.shadowRoot && b.shadowRoot.querySelector("#connect");
    if (inner) inner.click(); else if (b) b.click();
    document.dispatchEvent(new MouseEvent("mousemove", {bubbles: true}));
    console.log("[mct] keep-alive", new Date().toLocaleTimeString());
  }, 60 * 1000);
}
'''))
print("接続の維持を開始しました")


In [ ]:
#@title 4. 起動 (実行したままにしてください)
%cd /content/app
RESUME_ARG = "--resume latest" if AUTO_RESUME else ""
%run run.py --tunnel {TUNNEL} --workdir "{WORKDIR}" {RESUME_ARG}
